<a href="https://colab.research.google.com/github/ghadirchhade/Master-Thesis/blob/main/mAP50_image_level_differences_E02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

path_without_tiling = "/content/drive/MyDrive/master_thesis/evaluation/image_level_E02_1.csv"
path_with_tiling    = "/content/drive/MyDrive/master_thesis/evaluation/image_level_E02_2.csv"

df_without = pd.read_csv(path_without_tiling)
df_with    = pd.read_csv(path_with_tiling)

# Match on image_ID only
merged = df_without.merge(
    df_with,
    on="image_ID",
    suffixes=("_without_tiling", "_with_tiling"),
    how="inner"
)

# Warn if some images aren't present in both files
n_without, n_with, n_matched = len(df_without), len(df_with), len(merged)
if n_matched < max(n_without, n_with):
    missing_in_with = set(df_without["image_ID"]) - set(df_with["image_ID"])
    missing_in_without = set(df_with["image_ID"]) - set(df_without["image_ID"])
    print(f"⚠️ {n_without} images (without tiling), {n_with} images (with tiling), "
          f"only {n_matched} matched on image_ID.")
    if missing_in_with:
        print(f"  Images only in 'without tiling': {missing_in_with}")
    if missing_in_without:
        print(f"  Images only in 'with tiling': {missing_in_without}")

# Compute difference (per image)
merged["Difference_mAP50"] = (
    merged["mAP50_image_mean_without_tiling"] - merged["mAP50_image_mean_with_tiling"]
)

# Output dataframe
output_cols = [
    "image_ID",
    "mAP50_image_mean_without_tiling",
    "mAP50_image_mean_with_tiling",
    "Difference_mAP50"
]
diff_df = merged[output_cols].copy()

# Save
output_path = "/content/drive/MyDrive/master_thesis/evaluation/mAP50_image_level_differences_E02.csv"
diff_df.to_csv(output_path, index=False)
print(f"Saved differences to: {output_path}")

# Counts
# Difference > 0  -> without_tiling higher -> tiling did NOT improve
# Difference < 0  -> with_tiling higher    -> tiling DID improve
# Difference == 0 -> no change
n_tiling_improved     = (diff_df["Difference_mAP50"] < 0).sum()
n_tiling_not_improved = (diff_df["Difference_mAP50"] > 0).sum()
n_tiling_no_change    = (diff_df["Difference_mAP50"] == 0).sum()

print(f"Images where tiling improved results: {n_tiling_improved}")
print(f"Images where tiling did NOT improve results: {n_tiling_not_improved}")
print(f"Images with no change: {n_tiling_no_change}")

Saved differences to: /content/drive/MyDrive/master_thesis/evaluation/mAP50_image_level_differences_E02.csv
Images where tiling improved results: 25
Images where tiling did NOT improve results: 102
Images with no change: 9
